# Lecture 17 — pandas: Tables With Names
### EES 3350/5350 · Python in Earth Science · Wednesday, October 14, 2026

Last week you loaded a year of Nashville weather with `np.loadtxt()`, pulled out columns by **number** (`data[:, 2]` was the daily high... or was it column 3?), and counted days to find that Christmas was index 358.

Here's a confession: the file you used last week **wasn't** the file NOAA gave me. I cleaned it up first, because NumPy couldn't read the real one. Today you'll meet the real file, and the tool scientists actually use for tables like it: **pandas**.

**Learning objectives for today**
- explain why `np.loadtxt()` can't read most real data files, and read one with `pd.read_csv()`
- describe a **DataFrame**: its columns, its index, and its rows
- pull out one column (a **Series**) or several columns, and see the NumPy array underneath
- summarize a whole table at once with `.describe()` and `.info()`
- select rows by **position** with `.iloc[]` and by **label** with `.loc[]`
- use **dates** as row labels, so `"2025-12-25"` finds Christmas for you

---
## Part 0 : get today's files

In a terminal:
```
cd ~/course-materials
git pull
ls week08
```

You should see a new `week08` folder with this notebook and `nashville_ghcnd_2025.csv`. Copy the notebook into your own folder:
```
cp week08/17_pandas_intro.ipynb ~/LastName_EES3350/lectures/17_pandas_intro_LastName.ipynb
```

As in Lecture 15, the data file stays in `course-materials`, and we reach it with a **relative path** from your `lectures` folder.

In [ ]:
import numpy as np

raw_path = "../../course-materials/week08/nashville_ghcnd_2025.csv"

---
## Part 1 : the file NumPy couldn't read

This is the file exactly as it came from NOAA's Climate Data Online. Let's peek at the first 3 lines with the `!head` trick from Lecture 14:

In [ ]:
!head -3 ../../course-materials/week08/nashville_ghcnd_2025.csv

That's a lot. Each line is **one day**, but now there are **dozens of columns**, a header line with column **names**, values wrapped in `"quotes"`, a station **name** that contains a comma (`"NASHVILLE INTERNATIONAL AIRPORT, TN US"`), and a **date** written as text (`"2025-01-01"`).

*Data: NOAA National Centers for Environmental Information (NCEI), Global Historical Climatology Network – Daily (GHCN-Daily), station USW00013897 (Nashville Intl AP, TN), January 1 – December 31, 2025, ordered through [Climate Data Online](https://www.ncdc.noaa.gov/cdo-web/) in standard (US) units. It's the same station and year as Lectures 14 and 15.*

Let's try what we know:

In [ ]:
data = np.loadtxt(raw_path, delimiter=",")

#### Reading the error
Start at the bottom: a `ValueError`. NumPy *could not convert string `'"STATION"'` to float*.

`np.loadtxt()` assumes every value in the file is a **number**, because an array can only hold **one type** (Lecture 14). The very first thing it found was the word `"STATION"` in the header. Even if we skipped the header, the next row starts with the text `"USW00013897"`.

So last week, before you ever saw the file, I:
1. deleted the header line and wrote the column names in a `#` comment instead,
2. threw away every column except the 4 we needed,
3. split the text date `2025-01-01` into a month number and a day number,
4. removed all the quotes.

That's fine for one file. It is **not** fine for the hundreds of files you'll meet in research. We need a tool built for tables with **names, text, dates, and numbers mixed together**.

#### Enter pandas
**pandas** is Python's library for tables. It's built **on top of NumPy**, so everything you learned last week still applies underneath. 

Just like NumPy is always `np`, pandas is always imported as `pd`:

In [ ]:
import pandas as pd

df = pd.read_csv(raw_path)

df.head()

#### What just happened?
`pd.read_csv()` read the **whole** file, header and text and all, into a **DataFrame**: pandas' name for a table. We call it `df`, which is the conventional name for "the DataFrame I'm working with."

`df.head()` shows the first 5 rows. Notice what pandas figured out on its own:

|**the file has...**|**`np.loadtxt()`**|**`pd.read_csv()`**|
|:--|:--|:--|
|a header line with column names|crashes|uses them as **column names**|
|text values like `"USW00013897"`|crashes|keeps them as text|
|a comma *inside* quotes (`"NASHVILLE ..., TN US"`)|crashes|knows it's one value|
|empty values|crashes|fills them with `NaN` (missing)|
|numbers|reads them|reads them|

Those `NaN`s are the same `np.nan` from Lecture 15: pandas uses NumPy's "missing" marker.

#### When to use which?
pandas doesn't replace NumPy. They do different jobs, and you'll use both, often in the same notebook:

|**NumPy**|**pandas**|
|:--|:--|
|arrays of numbers, all one type|tables where each column can be a different type|
|positions: row 2, column 3|names: the `"TMAX"` column, the `"2025-12-25"` row|
|math: element-wise operations, broadcasting, `axis`|data handling: reading files, missing values, dates, summaries|
|gridded data: DEMs, model output, images|tabular data: anything you'd open in a spreadsheet|

##### <center>Think of a DataFrame as a spreadsheet you control with code: every column is a NumPy array with a name, lined up by row.</center>

---
## Part 2 : meet the DataFrame

A DataFrame has three parts:

```
                    columns: names, not numbers
               ┌────────────┬──────────┬──────┬──────┬─────┐
               │  STATION   │   DATE   │ PRCP │ TMAX │ ... │
       ┌───────┼────────────┼──────────┼──────┼──────┼─────┤
index  │   0   │ USW00013897│2025-01-01│ 0.00 │  44  │     │
(row   │   1   │ USW00013897│2025-01-02│ 0.00 │  49  │     │   ← values: each column
labels)│  ...  │    ...     │   ...    │ ...  │ ...  │     │     is a NumPy array
       │  364  │ USW00013897│2025-12-31│ 0.00 │  51  │     │
       └───────┴────────────┴──────────┴──────┴──────┴─────┘
```

- the **columns** have **names**,
- the **index** labels each row (for now just 0, 1, 2, ..., like array positions),
- the **values** inside are stored column by column, and **each column has its own type**. That's how a DataFrame can hold text in one column and numbers in the next, which an array never could.

A DataFrame knows about itself, just like an array:

In [ ]:
print(type(df))
print(df.shape)

`.shape` works exactly as it did for 2D arrays: **rows first, then columns**. 365 days, and **44** columns. Last week's file had 6. What are all of these?

In [ ]:
df.columns

#### Your first look at any new DataFrame
These are the commands to run every time you load a new file, before you try to analyze anything:

|**what you type**|**what it shows you**|
|:--|:--|
|`df.head()` / `df.head(10)`|the column names and the **first** 5 (or 10) rows|
|`df.tail()` / `df.tail(10)`|the column names and the **last** 5 (or 10) rows|
|`df.shape`|`(rows, columns)`, as a tuple|
|`len(df)`|the number of rows|
|`df.columns`|the column names|
|`df.dtypes`|the data type of every column|
|`df.info()`|all of the above at once, plus how many values in each column are **not** missing|
|`df.describe()`|summary statistics for every number column|

Notice the pattern from Lecture 14: no parentheses on **attributes** (`.shape`, `.columns`, `.dtypes`), which are facts the DataFrame already knows; parentheses on **methods** (`.head()`, `.info()`), which have to run.

`df.tail()` is a good habit: the end of a file is where you find out whether the data actually covers the dates you expect.

In [ ]:
df.tail(3)

The last row is December 31, 2025, so we have the whole year. Now the column types:

In [ ]:
df[["STATION", "DATE", "TMAX", "PRCP", "AWND", "TAVG"]].dtypes

`df.dtypes` lists the type of **every** column; here we picked 6 so the list fits on screen. The ones you'll see most:

|**dtype**|**holds**|**example column**|
|:--|:--|:--|
|`int64`|whole numbers|`TMAX`|
|`float64`|decimals, **and any number column with a missing value** (`NaN` is a float)|`PRCP`, `AWND`|
|`object`|text (or a mix of types)|`STATION`, `NAME`, `DATE`|
|`datetime64`|real dates and times|*(none yet: Part 4)*|

`TMAX` is `int64` because every high is a whole number and none are missing. `TAVG` is `float64` even though it holds no numbers at all: an all-`NaN` column is a float column.

NOAA uses short codes for every column. Here are the ones we'll use, from NOAA's GHCN-Daily documentation ([NCEI, readme.txt](https://www.ncei.noaa.gov/pub/data/ghcn/daily/readme.txt)):

|**column**|**what it holds**|**units (standard)**|**last week's file**|
|:--|:--|:--|:--|
|`DATE`|the day|text, `YYYY-MM-DD`|split into month, day|
|`TMAX`|daily high temperature|°F|column 2|
|`TMIN`|daily low temperature|°F|column 3|
|`PRCP`|precipitation|inches|column 4|
|`SNOW`|snowfall|inches|column 5|
|`SNWD`|snow depth on the ground|inches|*(not included)*|
|`AWND`|average daily wind speed|mph|*(not included)*|
|`WSF5`|fastest 5-second wind gust|mph|*(not included)*|
|`WT01`, `WT03`, ...|weather types: fog, thunder, sleet, ... (`1` = it happened)|—|*(not included)*|

Every measurement column also has an `_ATTRIBUTES` partner holding NOAA's codes about **how** it was measured and checked. We won't need those today.

`TAVG` is in the list too, but if you look at `df.head()`, it's all `NaN`: this station doesn't report a daily average. That's why in Lecture 14 we built our own as `(tmax + tmin) / 2`.

<div class="alert alert-info">

### Exercise 1 : getting oriented

1. Use `.shape` and an f-string to print `This file has 365 rows and 44 columns.` (Don't type the numbers! Index into the tuple.)
2. `df.columns` behaves like a list. Print the **first 6** column names with a slice.
3. Print `len(df)`. Which number from `.shape` does it match?

</div>

In [ ]:
# your code here
# 1
print(f"This file has {___} rows and {___} columns.")

# 2
print(___)

# 3
print(___)

---
## Part 3 : columns are Series

Last week, getting the daily highs meant remembering that they were column 2: `tmax = data[:, 2]`. In pandas you ask for a column **by name**, inside square brackets, like a dictionary key (Lecture 05):

In [ ]:
tmax = df["TMAX"]
print(tmax)
print("---")
tmax.head(7)

These are the same first-week highs you typed by hand in Lecture 14.

One column of a DataFrame is called a **Series**. A Series is a 1D array with **labels** attached: the numbers on the left are the index, and the numbers on the right are the values. The last line tells you the column's `Name` and its `dtype`.

In [ ]:
print(type(tmax))
print(tmax.dtype)

#### It's NumPy underneath
`.to_numpy()` hands you the array inside, the same kind of array as last week:

In [ ]:
tmax_array = tmax.to_numpy()

print(type(tmax_array))
print(tmax_array[:7])
print(tmax_array.shape)

And because a Series *is* an array with labels, everything from last week works on it directly: summary methods and element-wise math.

In [ ]:
print(f"Hottest high: {tmax.max()} F")
print(f"Average high: {tmax.mean():.1f} F")

tmax_c = (tmax - 32) * 5 / 9
tmax_c.head(3)

|**what you want**|**NumPy (last week)**|**pandas (today)**|
|:--|:--|:--|
|the daily highs|`data[:, 2]`, if you remember which column|`df["TMAX"]`|
|the biggest one|`tmax.max()`|`df["TMAX"].max()`|
|convert to °C|`(tmax - 32) * 5 / 9`|`(df["TMAX"] - 32) * 5 / 9`|
|the plain array|*(it already is one)*|`df["TMAX"].to_numpy()`|

#### Several columns at once
To pick **more than one** column, put a **list** of names inside the brackets. That's why there are two sets of brackets: the outer ones select, and the inner ones make the list.

In [ ]:
weather = df[["DATE", "TMAX", "TMIN", "PRCP", "SNOW", "SNWD"]].copy()

weather.head()

With one name you get a **Series**; with a list of names you get a smaller **DataFrame**. The `.copy()` at the end is Lecture 15's lesson about views: `weather` is our own independent table, so changing it later can never change `df`.

Here's what happens with only one set of brackets:

In [ ]:
df["TMAX", "TMIN"]

<div class="alert alert-danger">

**Common Error:** `df["TMAX", "TMIN"]` raises a `KeyError`. Without the inner brackets, pandas looks for **one** column whose name is the pair `("TMAX", "TMIN")`, and there isn't one. Read the bottom line of the error: it shows you the exact name pandas looked for. To select several columns, pass a **list**: `df[["TMAX", "TMIN"]]`. A misspelled name (`df["Tmax"]`) is a `KeyError` too: column names are case-sensitive.

|**you write**|**you get**|
|:--|:--|
|`df["TMAX"]`|one column, as a **Series**|
|`df[["TMAX"]]`|one column, as a **DataFrame** (a list with one name in it)|
|`df[["TMAX", "TMIN"]]`|several columns, as a **DataFrame**|
|`df["TMAX", "TMIN"]`|`KeyError`|

</div>

#### The whole table in one line
Remember spending a whole section of Lecture 14 computing the sum, mean, max, and min, one at a time? `.describe()` does all of that for **every number column at once**:

In [ ]:
weather.describe()

#### What just happened?
Each column got its own summary:

|**row**|**means**|**for `TMAX`**|
|:--|:--|:--|
|`count`|how many values are **not** missing|365|
|`mean`|average|73.3 °F, the same as `tmax.mean()`|
|`std`|standard deviation: how spread out the values are|17.7 °F|
|`min` / `max`|smallest / biggest|23 and 101|
|`25%`, `50%`, `75%`|**percentiles**: 25% of days were at or below the `25%` value, and so on. `50%` is the **median**.|`50%` = 77: half the days had a high of 77 °F or below|

`DATE` doesn't appear, because it's text and you can't average text. That brings us to the last summary tool:

In [ ]:
weather.info()

`.info()` lists every column with its **non-null count** (how many values aren't missing) and its **dtype**. Two things to notice:
1. Every column has 365 non-null values: this table has no gaps.
2. `DATE` has dtype `object`, which is pandas' word for **text** here. pandas doesn't know these are dates yet; to pandas, `"2025-12-25"` is just 10 characters. We'll fix that in Part 4.

#### One statistic at a time
`.describe()` is great for a first look. When you need **one** number, every summary method from Lecture 14 works on a column, plus a few more:

|**method**|**what it does**|
|:--|:--|
|`df["col"].sum()`|adds all the values|
|`df["col"].mean()`|average|
|`df["col"].median()`|middle value|
|`df["col"].min()`, `df["col"].max()`|smallest / largest value|
|`df["col"].std()`|standard deviation|
|`df["col"].idxmin()`, `df["col"].idxmax()`|the **index label** of the smallest / largest value (pandas' version of `.argmin()` / `.argmax()`)|
|`df["col"].count()`|how many values are **not** missing|

<div class="alert alert-info">

### Exercise 2 : was 2025 normal, the pandas way

In Lecture 14 you answered these with arrays. Answer them again using `weather`, one line each.

1. Total precipitation for the year. (Lecture 14 answer: 54.91 in. Format to 2 decimals so the floating-point crumb disappears.)
2. The average daily high and the coldest low of the year.
3. The **deepest snow on the ground** (`SNWD`) all year. This column wasn't in last week's file!
4. Use `.describe()` on **only** the `PRCP` column. What is its **median** (the `50%` row)? Why is it so much smaller than its **mean**?

</div>

In [ ]:
# your code here
# 1
print(f"Total precipitation: {___:.2f} in")

# 2
print(f"Average high: {___:.1f} F, coldest low: {___} F")

# 3
print(f"Deepest snow on the ground: {___} in")

# 4
___

---
## Part 4 : rows — `.iloc[]` by position, `.loc[]` by label

Square brackets on a DataFrame pick **columns**. To pick **rows**, pandas has two tools, and the difference between them is the most important idea today:

|**tool**|**selects by**|**think of it as**|
|:--|:--|:--|
|`.iloc[]`|**position**: 0, 1, 2, ...|**i**nteger **loc**ation, exactly like array indexing|
|`.loc[]`|**label**: whatever the index says|**loc**ate by name|

#### `.iloc[]`: positions, just like arrays
Everything you know about indexing arrays works inside `.iloc[]`, including `[row, col]` from Lecture 16:

In [ ]:
print(weather.iloc[0])

One row comes back as a **Series**, with the column names as its labels. Christmas is position 358 (Lecture 14's table), and the high is column position 1:

In [ ]:
print(weather.iloc[358, 1])
print("----------------")
print(weather.iloc[358])

Slices work too, with the stop **not** included:

In [ ]:
weather.iloc[0:7]

<div class="alert alert-warning">

**Help me out:** predict what each line prints, then run them. What's the difference?
```python
print(weather.iloc[358, 1])
print(weather.loc[358, "TMAX"])
```

</div>

In [ ]:
print(weather.iloc[358, 1])
print(weather.loc[358, "TMAX"])

#### `.loc[]`: labels
Both print 71. 

- `.iloc[358, 1]` means "**position** 358, **position** 1." 
- `.loc[358, "TMAX"]` means "the row **labeled** 358, the column **named** `TMAX`."

Right now, the index labels are 0, 1, 2, ..., so `.loc[358]` and `.iloc[358]` give the same row. That's a coincidence, and it hides why `.loc` is useful. The labels **should** be something meaningful. For a daily weather record, the obvious label for each row is its **date**.

Two steps:
1. tell pandas the `DATE` column holds dates, with `pd.to_datetime()`,
2. make it the index, with `.set_index()`.

In [ ]:
weather["DATE"] = pd.to_datetime(weather["DATE"])
weather = weather.set_index("DATE")

weather.head()

#### What just happened?
- `pd.to_datetime()` converted the text `"2025-01-01"` into a real **date**. Check `weather.index.dtype` below: it says `datetime64`, NumPy's date type.
- `.set_index("DATE")` moved that column out of the table and made it the **row labels**. That's why `DATE` now sits on the left, in bold, slightly lower than the column names, and why there are now 5 columns instead of 6.

In [ ]:
print(weather.index.dtype)
print(weather.shape)

Now ask for Christmas **by name**. No more counting days in January through November:

In [ ]:
weather.loc["2025-12-25"]

Notice the highs and lows print as `71.0` and `63.0`, not `71` and `63`. One row is a Series, and a Series holds **one dtype**, so the whole row became floats to make room for `PRCP` (the same "everything becomes one type" rule as arrays in Lecture 14).

And one value, `[row, col]` style, with a **label** for each:

In [ ]:
print(weather.loc["2025-12-25", "TMAX"])

#### Which day? `.idxmax()` returns the date
In Lecture 15, finding the hottest day took `.argmax()` to get a **position**, then `month[i]` and `day[i]` to turn that position into a date. Now the labels **are** dates, so `.idxmax()` hands you the date directly:

In [ ]:
print(weather["TMAX"].idxmax())
print(weather["TMIN"].idxmin())

August 19 and January 20, the same days as Lecture 15. (`.idxmin()` returns the **first** coldest night; January 22 tied it, just like `.argmin()` in Lecture 15.)

#### Dates understand months
Because the labels are real dates, `.loc` can take **part** of a date. `"2025-07"` means "every day in July 2025":

In [ ]:
july = weather.loc["2025-07"]

print(july.shape)
print(f"July average high: {july['TMAX'].mean():.1f} F")

That's the 94.0 °F from Lecture 15, where we needed `tmax[month == 7]` and a whole separate `month` column.

And date **ranges** work with a slice. Summer, June 1 through August 31:

In [ ]:
summer = weather.loc["2025-06-01":"2025-08-31"]

print(len(summer))
print(f"Summer average high: {summer['TMAX'].mean():.1f} F")

92 days, and the same 91.3 °F as Lecture 14's `tmax[151:243]`. But look closely at that slice: we wrote `"2025-08-31"` and got August 31.

<div class="alert alert-danger">

**Common Error:** `.loc` slices **include the stop**. `.iloc` slices (and every other slice in Python) **leave it out**.

|**slice**|**includes the stop?**|**summer is...**|
|:--|:--|:--|
|`weather.iloc[151:243]`|no|positions 151 to **242**|
|`weather.loc["2025-06-01":"2025-08-31"]`|**yes**|June 1 to Aug 31, **including** Aug 31|

This is on purpose: with labels, "June 1 through August 31" should mean what it says, and you can't always know what the "next" label after Aug 31 would be. Just remember: **positions leave out the stop, labels keep it.**

</div>

Now that the labels are dates, a **number** is no longer a label:

In [ ]:
weather.loc[358]

#### Reading the error
`KeyError: 358`. There's no row **labeled** 358 anymore; the labels are dates. When you want a **position**, use `.iloc[358]`. When you want a **label**, use `.loc["2025-12-25"]`. If you're ever unsure which one you have, print `weather.index[:3]` and look.

|**use `.iloc[]` when...**|**use `.loc[]` when...**|
|:--|:--|
|you want the first or last N rows (`.iloc[-7:]`)|you want specific columns **by name**|
|you're thinking in positions (row 5, column 3)|you're thinking in labels: a date, a station name|
|you'd have used array indexing last week|you want a range of dates, **both ends included**|

<div class="alert alert-info">

### Exercise 3 : finding days by date

Use `.loc` with dates for 1–4. No index counting!

1. Print the high and low on **July 4**.
2. Make `feb`, every row in **February** 2025. How many rows does it have, and what was February's average high?
3. In Lecture 15 we learned that a slow storm system soaked the Ohio Valley from **April 2 to April 6**. How much precipitation fell at the airport over those 5 days, both ends included?
4. How much snow fell on **January 10**?
5. Use `.iloc` to print the **last 7 rows** of `weather`. (Which tool, `.loc` or `.iloc`, makes "the last 7" easy? Why?)

</div>

In [ ]:
# your code here
# 1
print(___)

# 2
feb = ___
print(___)

# 3
print(f"April 2-6: {___:.2f} in")

# 4
print(___)

# 5
___

Basically:
```
                          To grab a column: df["column name"]
                     ┌────────────┬──────────┬──────┬──────┬─────┐
                     │  STATION   │   DATE   │ PRCP │ TMAX │ ... │
             ┌───────┼────────────┼──────────┼──────┼──────┼─────┤
to grab a    │   0   │ USW00013897│2025-01-01│ 0.00 │  44  │     │
rows         │   1   │ USW00013897│2025-01-02│ 0.00 │  49  │     │   
df.iloc[row] │  ...  │    ...     │   ...    │ ...  │ ...  │     │    
             │  364  │ USW00013897│2025-12-31│ 0.00 │  51  │     │
             └───────┴────────────┴──────────┴──────┴──────┴─────┘

            You can grab both with df.iloc[row col]
                                   df.loc[row, "column name"]
                            or if the row is called something else
                                   df.loc["index name", "column name"]

```

---
## Part 5 : the columns NumPy never saw

So far we've redone last week's work more comfortably. But I kept only the date and 4 measurements from the raw file's **44 columns** before you ever saw it. Let's use some of the others.

#### A shortcut: dates and index in `read_csv()`
Converting dates and setting the index is so common that `read_csv()` can do both while it reads the file. Let's reload the **whole** file this way:

In [ ]:
df = pd.read_csv(raw_path, parse_dates=["DATE"], index_col="DATE")

df.loc["2025-12-25", ["TMAX", "TMIN", "PRCP", "AWND"]]

- `parse_dates=["DATE"]` converts the `DATE` column to real dates,
- `index_col="DATE"` makes it the index.

From now on, this is how we'll load daily data. Notice the last line, too: `.loc[row, list of columns]` picks one day and several columns at once.

`AWND` is the average wind speed each day, and `WSF5` is the fastest 5-second gust:

In [ ]:
wind = df[["AWND", "WSF5"]]

wind.describe()

Look at the `count` row: `AWND` has **362** values, not 365. 

So three days are missing. 
- `.isna()` makes a boolean Series that is `True` where a value is missing (`NaN`)
- `.sum()` counts the `True`s, the same counting trick as `np.sum(mask)` in Lecture 15

In [ ]:
df["AWND"].isna().sum()

#### `NaN` doesn't break pandas the way it broke NumPy
In Lecture 15, one `np.nan` turned `.mean()` into `nan`, and you needed `np.nanmean()`. pandas **skips missing values by default**:

In [ ]:
print(f"pandas .mean():   {df['AWND'].mean():.2f} mph")
print(f"np.mean(array):   {np.mean(df['AWND'].to_numpy()):.2f} mph")
print(f"np.nanmean(array): {np.nanmean(df['AWND'].to_numpy()):.2f} mph")

<div class="alert alert-success">

**Tip:** pandas methods like `.mean()`, `.sum()`, `.max()`, and `.describe()` **skip** `NaN` automatically. That's convenient, but it means pandas won't warn you that data is missing. Check `.info()` or `.isna().sum()` first, so you know how many values each answer is really based on.

</div>

<div class="alert alert-info">

### Exercise 4 : from scratch

No blanks this time. Write each answer yourself.

1. What was the **fastest wind gust** of 2025, in mph, and on **what date**?
2. What was the **average wind speed** in **March**, and in **August**? Which month was windier?
3. How many days are missing a `PGTM` value? (`PGTM` is the time of the peak gust.) Based on that, would you use `PGTM` for anything?

</div>

In [ ]:
# your code here

<div class="alert alert-success">

**Tip:** `:%B %d` is a format spec for dates (full month name, then day), the same idea as `:.2f` for numbers. Printing the date on its own is fine too.
```python
print(f"{df['PRCP'].idxmax():%B %d}")
```
---
```
April 03
```
 
</div>

<div class="alert alert-custom" style="background-color:#e8daef; border:1px solid #d2b4de; color:#6c3483; padding:10px; border-radius:4px;">

### Bonus: is there a column for ice?

On Lecture 15's ice-storm question, you discovered that last week's file had no column for **ice**. The raw file has a set of **weather type** columns, `WT01` through `WT08`. Each one is `1` on days that weather happened and `NaN` on days it didn't. From NOAA's documentation ([NCEI, readme.txt](https://www.ncei.noaa.gov/pub/data/ghcn/daily/readme.txt)):

|**column**|**weather type**|
|:--|:--|
|`WT01`|fog, ice fog, or freezing fog|
|`WT02`|heavy fog|
|`WT03`|thunder|
|`WT04`|ice pellets, sleet, snow pellets, or small hail|
|`WT05`|hail|
|`WT06`|**glaze** or rime (ice coating surfaces)|
|`WT08`|smoke or haze|

`.count()` counts the values that **aren't** missing, so on these columns it counts the days each weather type happened.

1. On how many days in 2025 was **thunder** reported? **Fog**?
2. Use `.count()` on all seven `WT` columns at once (select them with a list).
3. How many days had **glaze**? Does that surprise you, given the ice storm happened in **January 2026**?
4. Which month of 2025 had the most thunder days? (Try `.loc` with each month, or wait until Friday for a much faster way.)

</div>

In [ ]:
# your code here

<details>
<summary style="color:green;"><strong>Answer.</strong></summary>

```python
# 1
print(f"Thunder days: {df['WT03'].count()}")
print(f"Fog days:     {df['WT01'].count()}")

# 2
print(df[["WT01", "WT02", "WT03", "WT04", "WT05", "WT06", "WT08"]].count())

# 4 (one month at a time, for now)
for m in ["2025-04", "2025-05", "2025-06", "2025-07"]:
    print(m, df.loc[m, "WT03"].count())
```
---
```
Thunder days: 61
Fog days:     140
WT01    140
WT02     14
WT03     61
WT04      2
WT05      1
WT06      0
WT08     62
dtype: int64
2025-04 5
2025-05 9
2025-06 11
2025-07 13
```

For #3: zero glaze days in 2025, which fits: the January 2026 ice storm isn't in this file. If we downloaded 2026, the `WT06` column is where it should show up. For #4, Friday's `groupby` will answer "which month?" for all 12 months in one line.

</details>

---
## A useful table

|**NumPy (last week)**|**pandas (today)**|
|:--|:--|
|`np.loadtxt(path, delimiter=",")`, numbers only|`pd.read_csv(path)`, names, text, dates, and numbers|
|`data[:, 2]`, by column **number**|`df["TMAX"]`, by column **name**|
|*(n/a)*|`df[["TMAX", "TMIN"]]`, several columns (a list!)|
|`a.shape`|`df.shape`, `df.head()`, `df.tail()`, `df.dtypes`|
|`.sum()`, `.mean()`, ... one at a time|`df.describe()`, all of them for every column|
|*(n/a)*|`df.info()`, dtypes and missing-value counts|
|`a[358]`, `a[358, 2]`|`df.iloc[358]`, `df.iloc[358, 1]` (positions)|
|count days to find index 358|`df.loc["2025-12-25"]` (labels)|
|`a.argmax()`, then `month[i]`, `day[i]`|`df["TMAX"].idxmax()` gives the date|
|`tmax[151:243]`, stop left out|`df.loc["2025-06-01":"2025-08-31"]`, stop **included**|
|`np.nanmean(a)` to skip `nan`|`df["AWND"].mean()` skips `NaN` automatically|
|`a.to_numpy()` *(n/a: it's already an array)*|`df["TMAX"].to_numpy()`|